In [2]:
import os
import pandas as pd
import numpy as np
import scipy.signal as signal

# Prepare for the Challenge 2015 dataset and filtering
with_ppg = []
for dirname, _, filenames in os.walk('Challenge2015_csv/'):
    for filename in filenames:
        if filename != "ALARMS":
            if "PLETH" in pd.read_csv(os.path.join(dirname, filename)).columns:
                with_ppg.append(filename[0:-4])

# Read ALARMS file and filter out asystole
df = pd.read_csv("Challenge2015_csv/ALARMS", names=["Sample", "Label", "TrueAlarm"])
df_relevant = df.loc[df["Label"] != "Asystole"].copy()

# Only include samples with PPG
df_filtered = df_relevant.loc[df_relevant["Sample"].isin(with_ppg)].copy()

# Add Path column for the CSV files
df_filtered["Path"] = ["Challenge2015_csv/" + x + ".csv" for x in df_filtered["Sample"]]

# Read MIMIC dataset for Atrial Fibrillation and Healthy data
afib_paths = []
healthy_paths = []
afib_samples = []
healthy_samples = []

for dirname, _, filenames in os.walk('mimic_perform_af_csv'):
    for filename in filenames:
        if "csv" in filename:
            afib_paths.append(os.path.join(dirname, filename))
            filename_split = filename.split("_")
            afib_samples.append(filename_split[2] + "_" + filename_split[3])

for dirname, _, filenames in os.walk('mimic_perform_non_af_csv'):
    for filename in filenames:
        if "csv" in filename:
            healthy_paths.append(os.path.join(dirname, filename))
            filename_split = filename.split("_")
            healthy_samples.append(filename_split[2] + "_" + filename_split[3] + "_" + filename_split[4])

# Combine all datasets
afib_df = pd.DataFrame({"Sample": afib_samples, "Label": 19 * ["Atrial_Fibrillation"], "TrueAlarm": 19 * [True], "Path": afib_paths})
healthy_df = pd.DataFrame({"Sample": healthy_samples, "Label": 16 * ["Healthy"], "TrueAlarm": 16 * [True], "Path": healthy_paths})

# Merge the datasets
df_filtered = pd.concat([df_filtered, afib_df, healthy_df])

# Function for downsampling
def downsample(signal_data, original_fs=250, target_fs=125):
    if original_fs == target_fs:
        return signal_data
    
    nyquist = target_fs / 2
    b, a = signal.butter(4, nyquist / (original_fs / 2), btype='low', analog=False)
    filtered_signal = signal.filtfilt(b, a, signal_data)
    downsampled_signal = filtered_signal[::2]
    return downsampled_signal

# Function for standardizing signals
def standardize(data):
    std_dev = np.std(data)
    if std_dev == 0:
        return data  # If std is zero, return the data unchanged
    return (data - np.mean(data)) / std_dev

# Function to get entire ECG and PPG waveforms without chunking
def getWaveforms(label, index, sampleType, trueAlarm=1):
    filtered_df = df_filtered.loc[(df_filtered["Label"] == label) & (df_filtered["TrueAlarm"] == trueAlarm)].reset_index(drop=True)
    
    if index >= len(filtered_df):  # Ensure index is within bounds
        return []

    file_path = filtered_df.loc[index, "Path"]  # Use the correct 'Path' column

    # Determine the ECG column based on the file
    try:
        # Read the header of the CSV file to check for available ECG columns
        file_data = pd.read_csv(file_path, nrows=1)
        # Check for possible ECG column labels
        ecg_column = None
        if "ECG" in file_data.columns:
            ecg_column = "ECG"
        elif "II" in file_data.columns:
            ecg_column = "II"
        elif "III" in file_data.columns:
            ecg_column = "III"
        else:
            print(f"Warning: No known ECG column found in {file_path}. Skipping this file.")
            return []
        
        # Read the full ECG waveform from the correct column
        full_waveform = pd.read_csv(file_path)[ecg_column].values
        
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return []  # Skip this file if an error occurs

    # Remove NaN values from the waveform
    full_waveform = full_waveform[~np.isnan(full_waveform)]  # Remove NaN values

    # If the waveform is empty after removing NaNs, skip this file
    if len(full_waveform) == 0:
        print(f"Warning: {ecg_column} data in {file_path} contains only NaN values after removal. Skipping this file.")
        return []

    # Standardize the data
    full_waveform = standardize(full_waveform)
    
    return full_waveform  # Return the full waveform, no chunking

# Extract ECG and PPG samples for dataset preparation
ecg_data = []
ppg_data = []
labels = []

for label in df_filtered["Label"].unique():
    for i in range(len(df_filtered[df_filtered["Label"] == label])):
        waveform_parts_ecg = getWaveforms(label, i, "ECG", trueAlarm=1)
        waveform_parts_ppg = getWaveforms(label, i, "PPG", trueAlarm=1)
        
        # Check if both waveforms are non-empty
        if len(waveform_parts_ecg) > 0 and len(waveform_parts_ppg) > 0:
            ecg_data.append(waveform_parts_ecg)  # Append the full waveform directly
            ppg_data.append(waveform_parts_ppg)  # Append the full waveform directly
            labels.append(label)  # Add the corresponding label for each entry

# Save the ECG and PPG files in the same file with the corresponding labels for later use
output_dir = "arrhythmia_data/"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Combine ECG and PPG data and save them in the same file for each patient
for i, (ecg, ppg, label) in enumerate(zip(ecg_data, ppg_data, labels)):
    # Ensure that the length of ecg and ppg match
    if len(ecg) != len(ppg):
        print(f"Warning: ECG and PPG lengths don't match for sample {label}_{i}. Skipping this entry.")
        continue
    
    # Combine ECG and PPG data into one matrix (columns for ECG and PPG)
    combined_data = np.column_stack((ecg, ppg))  # ECG and PPG as columns
    
    # Create a DataFrame with proper column names
    combined_df = pd.DataFrame(combined_data, columns=["ECG", "PPG"])
    
    # Save to a single file with label and index in the filename
    combined_df.to_csv(f"{output_dir}/{label}_{i}_combined.csv", index=False)

print("Data preparation complete. Combined ECG and PPG files have been saved.")


Data preparation complete. Combined ECG and PPG files have been saved.
